In [ ]:
import sys, os, glob, shutil, traceback
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
print("GPU:", torch.cuda.get_device_name(0), flush=True)
mods = [p for p in glob.glob("/kaggle/input/**/train_ce_large.py", recursive=True)]
src = sorted(mods, key=lambda p: "hardneg" not in p)[0]
os.makedirs("/kaggle/working/src", exist_ok=True)
for p in glob.glob(os.path.dirname(src) + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
if "hard-negatives" not in open("/kaggle/working/src/train_ce_large.py").read():
    raise SystemExit("код старый, --hard-negatives не поддерживается")
pack = os.path.dirname(glob.glob("/kaggle/input/**/item_texts.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/pack", exist_ok=True)
for p in glob.glob(pack + "/*"):
    d = "/kaggle/working/pack/" + os.path.basename(p)
    if not os.path.exists(d): os.symlink(p, d)
stage1 = os.path.dirname([p for p in glob.glob("/kaggle/input/**/model.safetensors", recursive=True)
                          if "pack" not in p and "ce_" not in p][0])
print("чекпоинт:", stage1, flush=True)
hn = {os.path.basename(p): p for p in glob.glob("/kaggle/input/**/hard_neg_*.parquet", recursive=True)}
print("негативы:", sorted(hn), flush=True)
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.train_ce_large import main
for tag, extra in [("hard_strict", ["--hard-negatives", hn["hard_neg_strict.parquet"]]),
                   ("base_strict", [])]:
    print("=" * 60 + f"\nКОНФИГУРАЦИЯ {tag}\n" + "=" * 60, flush=True)
    sys.argv = ["train_ce_large","--prepacked","/kaggle/working/pack","--holdout-fold","0",
                "--base-model","DeepPavlov/rubert-base-cased","--resume-from",stage1,
                "--max-train-pairs","1000","--human-epochs","2","--human-learning-rate","2e-5",
                "--batch-size","256","--max-length","256",
                "--output", f"/kaggle/working/{tag}"] + extra
    try: main()
    except Exception:
        traceback.print_exc(); print(f"{tag} УПАЛА", flush=True)
